# Часть 2 — второй агент: риск-менеджер спрашивает аналитика

Задача: решать, можно ли выдать клиенту кредит. Делают это два агента — как две команды в банке:

| агент | что умеет | что знает |
|---|---|---|
| 📊 **аналитик** (`analyst_agent.py`, уже запущен) | ходит в базу через MCP, пишет SQL | данные банка |
| ⚖️ **риск-менеджер** (собираем здесь) | решает по кредиту, считает платёж | кредитную политику. **Доступа к базе у него нет** |

```
вопрос → ⚖️ риск-менеджер ──A2A──▶ 📊 аналитик ──MCP──▶ база
              │ ◀──── факты ─────────┘
              └─▶ политика + расчёт платежа → решение
```

Зачем второй агент, а не ещё один инструмент? Разделение **знаний и прав**: базу видит только аналитик,
правила кредитования знает только риск-менеджер. Каждого можно развивать, проверять и выдавать права
отдельно — как разные команды.

**Перед запуском** должны работать (каждое — в своём терминале, из папки занятия):

```bash
docker compose up -d                  # база банка + Langfuse (один раз)
uv run python mcp_server.py           # терминал 1: MCP-сервер с инструментами → :8101
uv run python analyst_agent.py        # терминал 2: агент-аналитик по A2A     → :8102
```

In [1]:
import httpx

for name, url in [("MCP-сервер", "http://localhost:8101/mcp"), ("аналитик", "http://localhost:8102/.well-known/agent-card.json")]:
    try:
        httpx.get(url, timeout=3)
        print(f"✅ {name} работает")
    except httpx.ConnectError:
        print(f"❌ {name} не запущен — см. команды выше")

✅ MCP-сервер работает
✅ аналитик работает


## Шаг 1. Модель

In [2]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model=os.getenv("MODEL", "openai/gpt-4o-mini"), base_url="https://openrouter.ai/api/v1",
                 api_key=os.getenv("OPENROUTER_API_KEY"))

## Шаг 2. Аналитик как инструмент

Риск-менеджер будет обращаться к аналитику через обычный `@tool`. Внутри — A2A-запрос из части 1.
Для модели риск-менеджера это просто инструмент «спросить аналитика»; что на той стороне целый
агент — ей знать не нужно.

In [3]:
import uuid

import httpx
from a2a.client import ClientConfig, create_client
from a2a.helpers.proto_helpers import get_artifact_text, get_message_text, new_text_part
from a2a.types import Message, Role, SendMessageRequest, TaskState
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool


async def send_to_analyst(question: str, metadata: dict | None = None) -> str:
    """One A2A request: a message to the analyst agent → the text of its answer."""
    client = await create_client("http://localhost:8102", client_config=ClientConfig(
        streaming=False, httpx_client=httpx.AsyncClient(timeout=300)))   # the analyst may think a while
    message = Message(role=Role.ROLE_USER, message_id=str(uuid.uuid4()), parts=[new_text_part(question)])
    if metadata:
        message.metadata.update(metadata)                # extra fields: here — the user's session id
    try:
        async for event in client.send_message(SendMessageRequest(message=message)):
            task = event.task
            if task.status.state == TaskState.TASK_STATE_COMPLETED:
                return get_artifact_text(task.artifacts[0])          # the answer
            return f"Аналитик не ответил: {get_message_text(task.status.message)}"   # e.g. failed
    finally:
        await client.close()


@tool
async def ask_analyst(question: str, config: RunnableConfig) -> str:
    """Ask the bank's analyst agent a factual question about the bank data (accounts, transactions,
    balances, loans). Ask concrete questions with the account number and the period."""
    session_id = config.get("configurable", {}).get("thread_id")    # our conversation = the user's session
    print(f"   📨 → аналитику: {question}")
    # we pass the session id only as a label for Langfuse — the analyst does not remember anything
    answer = await send_to_analyst(question, {"session_id": session_id} if session_id else None)
    print(f"   📨 ← аналитик: {answer[:300]}")
    return answer

`config` подставляет LangChain сам — модель этот параметр не видит (в схеме инструмента только
`question`). Из него берём номер текущего разговора и отправляем аналитику как **метку сессии** —
зачем, разберём в конце блокнота и в части 3.

Проверим инструмент руками:

In [4]:
await ask_analyst.ainvoke({"question": "Какой средний месячный приход на счёт 2378 за 1998 год?"})

   📨 → аналитику: Какой средний месячный приход на счёт 2378 за 1998 год?
   📨 ← аналитик: Средний месячный приход на счёт 2378 за 1998 год — **30 579,92 CZK**. Сумма рассчитана как общий объём входящих операций за год, делённый на 12.


'Средний месячный приход на счёт 2378 за 1998 год — **30\u202f579,92 CZK**. Сумма рассчитана как общий объём входящих операций за год, делённый на 12.'

## Шаг 3. Свой инструмент риск-менеджера — платёж по кредиту

Доверять модели расчёт платежа «в уме» нельзя — пусть считает код.

In [5]:
@tool
def loan_payment(amount: float, months: int, annual_rate_percent: float = 8.0) -> str:
    """Monthly payment of an annuity loan: amount in CZK, term in months, annual rate in percent."""
    rate = annual_rate_percent / 100 / 12
    payment = amount * rate / (1 - (1 + rate) ** -months)
    return f"{payment:.2f} CZK в месяц"


loan_payment.invoke({"amount": 150_000, "months": 24})

'6784.09 CZK в месяц'

## Шаг 4. Риск-менеджер

Кредитная политика — в системном промпте. Граф — готовый `create_agent` из занятия 1
(тот же «агент ⇄ инструменты»).

In [6]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

POLICY = """Ты риск-менеджер банка. Решаешь, можно ли выдать кредит клиенту (счёту).
Доступа к базе у тебя нет: все факты о счёте спрашивай у аналитика (ask_analyst) — конкретными
вопросами с номером счёта и периодом. Платёж считай инструментом loan_payment.

Кредитная политика (учебная). Суммы в кронах (CZK), ставка 8% годовых:
1. Ежемесячный платёж по новому кредиту — не больше 30% среднего месячного прихода на счёт за 1998 год.
2. Если у счёта есть кредит в статусе running_in_debt или finished_default — отказ. Кредитную
   историю проверяй за ВСЁ время, а не только за 1998 год: старый кредит тоже считается.
3. Средний баланс счёта за 1998 год должен быть не меньше одного ежемесячного платежа.

Ответ: решение ОДОБРИТЬ или ОТКАЗАТЬ, затем коротко по каждому пункту политики — цифры и вывод."""

risk_manager = create_agent(llm, [ask_analyst, loan_payment], system_prompt=POLICY,
                            checkpointer=InMemorySaver())       # the risk manager remembers the chat


async def decide(question: str, chat: str) -> None:
    """One message in the chat `chat` (= thread_id: a separate memory for each conversation)."""
    print("❓", question)
    result = await risk_manager.ainvoke({"messages": [("user", question)]},
                                        {"configurable": {"thread_id": chat}})
    print("\n⚖️", result["messages"][-1].text)

## Шаг 5. Решаем

Три клиента. Следите и здесь (📨 — разговор агентов), и в терминалах: аналитик получает вопросы,
MCP-сервер выполняет SQL. Риск-менеджер часто задаёт **несколько вопросов сразу** — аналитик
отвечает на них параллельно.

In [7]:
await decide("Можно ли выдать счёту 2378 кредит 150 000 CZK на 24 месяца?", chat="client-2378")

❓ Можно ли выдать счёту 2378 кредит 150 000 CZK на 24 месяца?
   📨 → аналитику: Для счёта 2378 укажи средний месячный приход за период 01.01.1998–31.12.1998 в CZK и, если возможно, расчёт/сумму прихода за этот период.
   📨 → аналитику: Для счёта 2378 укажи средний баланс за период 01.01.1998–31.12.1998 в CZK.
   📨 → аналитику: Для счёта 2378 проверь кредитную историю за всё доступное время: есть ли кредит(ы) в статусе running_in_debt или finished_default? Перечисли статусы и даты/периоды, если есть.
   📨 ← аналитик: За 1998 год по счёту 2378 зафиксировано **31 поступление** на общую сумму **366 959 CZK**. Средний месячный приход за 12 месяцев — **30 579,92 CZK** (сумма поступлений ÷ 12).
   📨 ← аналитик: По счёту 2378 кредитов со статусом `running_in_debt` или `finished_default` не найдено. Дат и периодов таких кредитов нет.
   📨 ← аналитик: Средний дневной баланс счёта 2378 за 01.01.1998–31.12.1998: **46 814,81 CZK**.

⚖️ **ОДОБРИТЬ**

1. **Платёж относительно прихода:** ежемесячный п

In [8]:
await decide("Можно ли выдать счёту 1583 кредит 300 000 CZK на 36 месяцев?", chat="client-1583")

❓ Можно ли выдать счёту 1583 кредит 300 000 CZK на 36 месяцев?
   📨 → аналитику: Для счёта 1583 укажи средний месячный приход за период 01.01.1998–31.12.1998 (CZK), а также расчёт/данные, на основе которых он определён.
   📨 → аналитику: Для счёта 1583 укажи средний баланс за период 01.01.1998–31.12.1998 (CZK).
   📨 → аналитику: Для счёта 1583 перечисли за всё время кредитную историю: все кредиты и их статусы, отдельно отметь наличие статусов running_in_debt или finished_default.
   📨 ← аналитик: Средний баланс счёта 1583 за 1998 год: **62 751,87 CZK**.
   📨 ← аналитик: По счёту **1583** найден 1 кредит:

| Кредит | Дата выдачи | Сумма | Срок | Ежемесячный платёж | Статус |
|---|---|---:|---:|---:|---|
| 5282 | 24.08.1996 | 388 512 CZK | 48 мес. | 8 094 CZK | `running_in_debt` |

Статус `running_in_debt` присутствует. Статусов `finished_default` нет.
   📨 ← аналитик: Средний месячный приход по счёту **1583** за 1998 год — **69 944,42 CZK**.

Расчёт: сумма всех операций с направлением `

In [9]:
await decide("Можно ли выдать счёту 1 кредит 100 000 CZK на 24 месяца?", chat="client-1")

❓ Можно ли выдать счёту 1 кредит 100 000 CZK на 24 месяца?
   📨 → аналитику: Для счёта 1 укажи общий приход за период 01.01.1998–31.12.1998 и средний месячный приход за этот период (CZK).
   📨 → аналитику: Для счёта 1 укажи средний баланс за период 01.01.1998–31.12.1998 (CZK).
   📨 → аналитику: Для счёта 1 перечисли кредиты за всё время: статус каждого, в частности есть ли running_in_debt или finished_default.
   📨 ← аналитик: Для счёта 1 кредитов нет. Статусов `running_in_debt` или `finished_default` не обнаружено.
   📨 ← аналитик: Счёт 1, 01.01.1998–31.12.1998:

- Общий приход: **45 416 CZK**
- Средний месячный приход: **3 784,67 CZK** (общая сумма ÷ 12)
   📨 ← аналитик: Средний дневной баланс счёта 1 за период 01.01.1998–31.12.1998: **15 195,52 CZK**.

⚖️ **ОТКАЗАТЬ.**

1. **Платёж:** 4 522,73 CZK в месяц. Лимит — 30% от среднего месячного прихода: 30% × 3 784,67 = **1 135,40 CZK**. Платёж превышает лимит.
2. **Кредитная история:** кредитов нет; статусов `running_in_debt` и `finishe

Ожидаемо: **2378 — одобрить** (платёж ~6,8 тыс. при приходе ~30,6 тыс.), **1583 — отказать**
(есть кредит `running_in_debt`), **1 — отказать** (приход ~3,8 тыс. — платёж больше 30%).

## Шаг 6. Уточняющий вопрос — память риск-менеджера

Продолжаем разговор про клиента 2378. Номер счёта не называем — риск-менеджер помнит его сам:

In [10]:
await decide("А если сумма 300 000 на тот же срок?", chat="client-2378")

❓ А если сумма 300 000 на тот же срок?

⚖️ **ОТКАЗАТЬ**

1. **Платёж относительно прихода:** платёж за 300 000 CZK на 24 месяца — **13 568,19 CZK в месяц**. Лимит — 30% от среднего месячного прихода 30 579,92 CZK, то есть **9 173,98 CZK**. Условие не выполнено.
2. **Кредитная история:** кредитов в статусах `running_in_debt` или `finished_default` не найдено. Условие выполнено.
3. **Средний баланс:** **46 814,81 CZK**, что больше ежемесячного платежа. Условие выполнено.


Риск-менеджер понял, о каком счёте речь, и **сам** сформулировал для аналитика полный вопрос
с номером счёта: аналитику прошлый разговор не нужен.

## Память в системе из нескольких агентов

Кажется удобным дать всем агентам один номер разговора — пусть все помнят всё. Так делать **не надо**:

- **параллельные вопросы портят историю**: риск-менеджер часто спрашивает аналитика о трёх вещах
  сразу — три запуска графа писали бы в одну ветку памяти одновременно и перемешали бы сообщения;
- **чужие разговоры сольются**: если другой агент (скажем, «отчётность») возьмёт тот же номер,
  аналитик смешает два чужих диалога в один;
- **агенты станут зависимы**: заказчику пришлось бы знать, как исполнитель хранит память, — а по
  смыслу A2A снаружи агент — чёрный ящик.

Как правильно — **два разных номера**:

| что | чей | зачем |
|---|---|---|
| **номер сессии** пользователя (`session_id`) | один на всю цепочку агентов | **связывать**: трейсы и логи одного обращения (часть 3) |
| **номер памяти** (`thread_id`) | у каждого агента свой | **помнить**: каждый агент решает сам |

У нас: риск-менеджер общается с человеком — у него память (`thread_id` = чат). Аналитик — исполнитель
**без памяти**: получает самодостаточные вопросы («счёт 2378, приход за 1998») и отвечает на каждый
отдельно. Номер сессии ему передаётся только как метка в `metadata` — для журнала, не для памяти.

> Если исполнителю всё же нужна память, в A2A для этого есть `contextId`: его выдаёт **сам
> исполнитель** в ответе, а заказчик хранит у себя «моя сессия → его contextId» и присылает этот
> номер в следующих сообщениях. Номера выдаёт исполнитель — значит, чужие разговоры не смешаются.

## Итоги

- второй агент обращается к первому по A2A так же, как к инструменту: для его модели это `@tool`;
- у каждого агента — свои знания, права, инструменты **и память**; общаются только через сообщения;
- помнит разговор тот, кто говорит с человеком; исполнитель отвечает на самодостаточные вопросы;
- агента-аналитика может вызывать кто угодно: другой агент, сервис, блокнот — ему всё равно.

Этот же риск-менеджер собран в файле `risk_agent.py` — в частях 3 и 4 мы его импортируем.

**Но что происходило внутри?** Сколько вызовов LLM, сколько токенов, где агент ошибся?
Из print-ов этого не понять. Дальше — часть 3: Langfuse.